In [ ]:
///- --package ..
:< sm'_operators
:< engine

#### site (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:== file = { path : string; content : string }

:= layout (root : string) (title : string) (body : string) : string =
    "<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
    ++# "<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">"
    ++# "<title>" ++# html.escape title ++# "</title>"
    ++# "<link rel=\"stylesheet\" href=\"" ++# root ++# "style.css\"></head>"
    ++# "<body><header><a class=\"home\" href=\"" ++# root ++# "\">Spiral Blog</a></header><main>"
    ++# body ++# "</main></body></html>"

:= entry_file (root : string) (e : page.entry) : file =
    { path = e.slug ++# "/index.html"; content = layout root e.title (page.render_entry e) }

:= ^ entry_files (root : string) (entries : list page.entry) : list file =
    ?= entries ?|
    | [] => []
    | e :: rest => entry_file root e :: entry_files root rest

:= base_url () : string = "https://i574n.github.io/blog"

:= files_at (root : string) (url : string) (entries : list page.entry) : list file =
    := index_page = { path = "index.html"; content = layout root "Spiral Blog" (index.render_index root entries) }
    := stylesheet = { path = "style.css"; content = style.css () }
    := feed = { path = "rss.xml"; content = rss.render_feed "Spiral Blog" url entries }
    := articles_page =
        {
            path = "articles/index.html"
            content = layout root "Articles - Spiral Blog" (index.render_index root (index.only page.Article entries))
        }
    := releases_page =
        {
            path = "releases/index.html"
            content = layout root "Releases - Spiral Blog" (index.render_index root (index.only page.Release entries))
        }
    index_page :: stylesheet :: feed :: articles_page :: releases_page :: entry_files root entries

:= files (entries : list page.entry) : list file = files_at "/blog/" (base_url ()) entries

:= ^ paths (fs : list file) : string =
    ?= fs ?|
    | [] => ""
    | [ f ] => f.path
    | f :: rest => f.path ++# " " ++# paths rest

#### the_site_has_an_index_and_one_folder_per_post (test)

In [1]:
///- --test --symbol
///> zig

files [ page.sample () ]
|> paths
|> _assert_eq "index.html style.css rss.xml articles/index.html releases/index.html sample/index.html"

__assert_eq { expected = index.html style.css rss.xml articles/index.html releases/index.html sample/index.html }

#### every_page_shares_the_layout (test)

In [2]:
///- --test --symbol
///> zig

:= ^ all_wrapped (fs : list file) : bool =
    ?= fs ?|
    | [] => !+
    | f :: rest ?& f.path |> sm'.ends_with ".html" =>
        (f.content |> sm'.starts_with "<!doctype html>")
        && (f.content |> sm'.ends_with "</main></body></html>")
        && all_wrapped rest
    | _ :: rest => all_wrapped rest
files [ page.sample () ] |> all_wrapped |> _assert_eq !+

__assert_eq { expected = true }

#### every_link_starts_at_the_site_root (test)

In [3]:
///- --test --symbol
///> zig

:= ^ links_at (want : string) (fs : list file) : i32 =
    ?= fs ?|
    | [] => 0i32
    | f :: rest => (? f.content |> sm'.contains want ?> 1i32 |: 0i32) + links_at want rest
:= fs = files_at "/x/" "https://h/x" [ page.sample () ]
(links_at "href=\"/x/style.css\"" fs, links_at "href=\"/style.css\"" fs, links_at "href=\"/\"" fs)
|> _assert_eq (4i32, 0i32, 0i32)

__assert_eq { expected = (4, 0, 0) }

#### the_post_page_title_is_escaped_into_the_head (test)

In [4]:
///- --test --symbol
///> zig

:= e = page.sample ()
files [ { e ?| title = "A & B" } ]
|> \ fs =>
    ?= fs ?|
    | _ :: _ :: _ :: _ :: _ :: post_file :: _ =>
        post_file.content |> sm'.contains "<title>A &amp; B</title>" |> _assert_eq !+
    | _ => failwith "expected a post file"

__assert_eq { expected = true }